In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.insert(0, "..")

In [ ]:
from pathlib import Path

import altair as alt
import geopandas as gpd
import osmnx as ox
import pandas as pd

In [ ]:
from main import OUTPUT_COLUMNS, Place, prepare_data_for_place
from src.route import (
    compute_routes_from_census_blocks_to_all_schools,
    compute_routes_from_census_blocks_to_school,
    get_route_gdf,
)

## Prepare out dir

In [ ]:
import shutil

OUT_PATH = Path("../data/out/notebook/")

# Delete the directory if it exists
if OUT_PATH.exists():
    shutil.rmtree(OUT_PATH)

# Recreate the directory
OUT_PATH.mkdir(parents=True, exist_ok=True)

## Load bike network

In [ ]:
place = Place(name="Somerville, MA, USA", residential_default_mph=20)
nodes, edges = prepare_data_for_place(place)

In [ ]:
edges = edges[OUTPUT_COLUMNS]

In [ ]:
G = ox.graph_from_gdfs(nodes, edges)

In [ ]:
edges.sample(3)

## Load Schools

In [ ]:
school_gdb_path = "../data/raw/SafeRoutesGISLayers.gdb.zip"

In [ ]:
layers = gpd.list_layers(school_gdb_path)
layers

In [ ]:
# read school data
schools_gdf = gpd.read_file(school_gdb_path, layer="PublicSchools")

# save schools polygons
schools_gdf.to_file((OUT_PATH / "schools_poly.gpkg"), driver="GPKG")

# make geom col into centroids
schools_gdf["geometry"] = schools_gdf.centroid

# save schools polygons
schools_gdf.to_file((OUT_PATH / "schools_centroid.gpkg"), driver="GPKG")

In [ ]:
schools_gdf.head(3)

## Load census blocks

In [ ]:
# read census blocks
census_blocks = gpd.read_file("../data/raw/Census_2020_Blocks.zip")

# filter by TOWN attribute
somerville_census_blocks = census_blocks[census_blocks["TOWN"] == "SOMERVILLE"].copy()

# reset index
somerville_census_blocks = somerville_census_blocks.reset_index(drop=True)

# get a sample
half_n_census_blocks = len(somerville_census_blocks) // 4
somerville_census_sample = somerville_census_blocks.sample(half_n_census_blocks)

# save polygon version
somerville_census_blocks.to_file((OUT_PATH / "somer_blocks_poly.gpkg"), driver="GPKG")
somerville_census_sample.to_file((OUT_PATH / "somer_sample_poly.gpkg"), driver="GPKG")

# convert geometry to centroid
somerville_census_blocks["geometry"] = somerville_census_blocks.centroid
somerville_census_sample["geometry"] = somerville_census_sample.centroid

# save centroid version
somerville_census_blocks.to_file(
    (OUT_PATH / "somer_blocks_centroid.gpkg"), driver="GPKG"
)
somerville_census_sample.to_file(
    (OUT_PATH / "somer_sample_centroid.gpkg"), driver="GPKG"
)

In [ ]:
somerville_census_blocks.head(3)

## Make sure everything has same crs

- EPSG:26986 =  NAD83 / Massachusetts Mainland Meters
- EPSG:4326 = WGS 84 / web

In [ ]:
def crs_first_line(gdf):
    return str(gdf.crs).splitlines()[0]

In [ ]:
print("somerville_census_blocks:", crs_first_line(somerville_census_blocks))
print("somerville_census_sample:", crs_first_line(somerville_census_sample))
print("schools_gdf             :", crs_first_line(schools_gdf))
print("edges                   :", crs_first_line(edges))
print("nodes                   :", crs_first_line(nodes))

In [ ]:
# use this one
use_crs = edges.crs

# make them match
somerville_census_blocks = somerville_census_blocks.to_crs(use_crs)
somerville_census_sample = somerville_census_sample.to_crs(use_crs)
schools_gdf = schools_gdf.to_crs(use_crs)
nodes = nodes.to_crs(use_crs)
edges = edges.to_crs(use_crs)

In [ ]:
print("somerville_census_blocks:", crs_first_line(somerville_census_blocks))
print("somerville_census_sample:", crs_first_line(somerville_census_sample))
print("schools_gdf             :", crs_first_line(schools_gdf))
print("edges                   :", crs_first_line(edges))
print("nodes                   :", crs_first_line(nodes))

## Routing

Try routing from a single block centroid to a single school

In [ ]:
G = ox.graph_from_gdfs(nodes, edges)

In [ ]:
# pick a school
dest_point = schools_gdf.loc[0, "geometry"]

In [ ]:
# pick a census centroid
orig_point = somerville_census_sample.iloc[0]["geometry"]

In [ ]:
# route based on composite score
route_gdf = get_route_gdf(G, orig_point, dest_point, weight="composite_score")
route_gdf

In [ ]:
# route based on length
route_gdf = get_route_gdf(G, orig_point, dest_point, weight="length")
route_gdf

## Route loop

Try routing from all block centroids to a single school

In [ ]:
# loop over all census blocks, computing routes to one school
combined_gdf, errors = compute_routes_from_census_blocks_to_school(
    G, somerville_census_blocks, schools_gdf.loc[0], weight="composite_score"
)

In [ ]:
combined_gdf.head(3)

In [ ]:
# save only the combined file
combined_gdf.to_file(OUT_PATH / "routes_to_school_0_by_risk.gpkg", driver="GPKG")

## Loop all

Route all block centroids to all schools

### Loop all schools - by risk

In [ ]:
all_routes_gdf_by_risk, errors = compute_routes_from_census_blocks_to_all_schools(
    G,
    somerville_census_blocks=somerville_census_blocks,
    schools_gdf=schools_gdf,
    weight="composite_score",
)

In [ ]:
# save only the combined file
all_routes_gdf_by_risk.to_file(OUT_PATH / "routes_by_risk.gpkg", driver="GPKG")

In [ ]:
all_routes_gdf_by_length, errors = compute_routes_from_census_blocks_to_all_schools(
    G,
    somerville_census_blocks=somerville_census_blocks,
    schools_gdf=schools_gdf,
    weight="length",
)

In [ ]:
# save only the combined file
all_routes_gdf_by_length.to_file(OUT_PATH / "routes_by_length.gpkg", driver="GPKG")

## Summarize

Combine into one df, drop geometry

In [ ]:
all_routes_gdf_by_length["method"] = "length"
all_routes_gdf_by_risk["method"] = "risk"

all_routes_gdf = pd.concat(
    [
        all_routes_gdf_by_length.drop(columns=["geometry"]),
        all_routes_gdf_by_risk.drop(columns=["geometry"]),
    ]
)

In [ ]:
school_means = (
    all_routes_gdf.groupby(["school_name", "method"])
    .mean(numeric_only=True)
    .reset_index()
)
school_means

#### Sort by difference

- median_composite_score: $length - risk$

In [ ]:
# reset
school_means = (
    all_routes_gdf.groupby(["school_name", "method"])
    .mean(numeric_only=True)
    .reset_index()
)

# pivot to get length and risk methods as separate columns
pivoted = school_means.pivot(
    index="school_name", columns="method", values="weighted_mean_score"
).reset_index()

# calculate the difference (length - risk)
pivoted["mean_score_diff"] = pivoted["length"] - pivoted["risk"]

# merge back with the original data
school_means = school_means.merge(
    pivoted[["school_name", "mean_score_diff"]], on="school_name"
).sort_values("mean_score_diff", ascending=False)

school_means.head()

In [ ]:
school_means.to_csv(OUT_PATH / "school_means.csv")

In [ ]:
# Create the bar chart with sorting by mean_score_diff
chart = (
    alt.Chart(school_means)
    .mark_bar()
    .encode(
        x=alt.X(
            "school_name:N",
            title="School",
            axis=alt.Axis(labelAngle=-45),
            sort=alt.EncodingSortField(
                field="mean_score_diff", op="max", order="descending"
            ),
        ),
        y=alt.Y("weighted_mean_score:Q", title="Weighted Mean Score"),
        color=alt.Color("method:N", title="Method", scale=alt.Scale(scheme="set2")),
        xOffset="method:N",  # This creates grouped bars
        tooltip=[
            alt.Tooltip("school_name:N", title="School"),
            alt.Tooltip("method:N", title="Method"),
            alt.Tooltip(
                "weighted_mean_score:Q", title="Weighted Mean Score", format=".3f"
            ),
            alt.Tooltip("sum_length:Q", title="Total Length", format=".1f"),
        ],
    )
    .properties(
        width=600,
        height=400,
        title="Mean Composite Score by School and Method (Risk vs. Length)",
    )
)

# Get one row per school for the diff labels (take the first method's row since diff is the same)
school_diff = (
    school_means.groupby("school_name")
    .agg(
        {
            "mean_score_diff": "first",
            "weighted_mean_score": "max",  # Use max to position text above tallest bar
        }
    )
    .reset_index()
)

# Add text labels centered above both bars showing mean_score_diff
text = (
    alt.Chart(school_diff)
    .mark_text(align="center", baseline="bottom", dy=-5, fontSize=10, fontWeight="bold")
    .encode(
        x=alt.X(
            "school_name:N",
            sort=alt.EncodingSortField(
                field="mean_score_diff", op="max", order="descending"
            ),
        ),
        y=alt.Y("weighted_mean_score:Q"),
        text=alt.Text("mean_score_diff:Q", format=".3f"),
    )
)

# Combine the chart and text
final_chart = chart + text
final_chart

In [ ]:
# save HTML to upload dir
final_chart.save(OUT_PATH / "chart1.html")

### Compare blocks

In [ ]:
block_means = (
    all_routes_gdf.groupby(["from_block_geoid", "method"])
    .mean(numeric_only=True)
    .reset_index()
)
block_means.head()

#### Get difference

In [ ]:
# get difference
block_means = (
    all_routes_gdf.groupby(["from_block_geoid", "method"])
    .mean(numeric_only=True)
    .reset_index()
)

# pivot to get length and risk methods as separate columns
pivoted = block_means.pivot(
    index="from_block_geoid", columns="method", values="weighted_mean_score"
).reset_index()

# calculate the difference (length - risk)
pivoted["mean_score_diff"] = pivoted["length"] - pivoted["risk"]

# merge back with the original data
block_means = block_means.merge(
    pivoted[["from_block_geoid", "mean_score_diff"]], on="from_block_geoid"
).sort_values("mean_score_diff", ascending=False)

block_means.head()

In [ ]:
# combine top 3 and bottom 3
top_three = pd.DataFrame(block_means.head(6))
top_three["class"] = "top_three"

bottom_three = pd.DataFrame(block_means.tail(6))
bottom_three["class"] = "bottom_three"

block_means_short = pd.concat([top_three, bottom_three])

In [ ]:
# save to csv
block_means_short.to_csv(OUT_PATH / "block_means_short.csv")

In [ ]:
# Create the bar chart with sorting by mean_score_diff
chart = (
    alt.Chart(block_means_short)
    .mark_bar()
    .encode(
        x=alt.X(
            "from_block_geoid:N",
            title="Block",
            axis=alt.Axis(labelAngle=-45),
            sort=alt.EncodingSortField(
                field="mean_score_diff", op="max", order="descending"
            ),
        ),
        y=alt.Y("weighted_mean_score:Q", title="Weighted Mean Score"),
        color=alt.Color("method:N", title="Method", scale=alt.Scale(scheme="set2")),
        xOffset="method:N",  # This creates grouped bars
        tooltip=[
            alt.Tooltip("from_block_geoid:N", title="Block"),
            alt.Tooltip("method:N", title="Method"),
            alt.Tooltip(
                "weighted_mean_score:Q", title="Weighted Mean Score", format=".3f"
            ),
            alt.Tooltip("sum_length:Q", title="Total Length", format=".1f"),
        ],
    )
    .properties(
        width=600,
        height=400,
        title="Mean Composite Score by Block and Method (Risk vs. Length)",
    )
)

# Get one row per block for the diff labels (take the first method's row since diff is the same)
block_diff = (
    block_means_short.groupby("from_block_geoid")
    .agg(
        {
            "mean_score_diff": "first",
            "weighted_mean_score": "max",  # Use max to position text above tallest bar
        }
    )
    .reset_index()
)

# Add text labels centered above both bars showing mean_score_diff
text = (
    alt.Chart(block_diff)
    .mark_text(align="center", baseline="bottom", dy=-5, fontSize=10, fontWeight="bold")
    .encode(
        x=alt.X(
            "from_block_geoid:N",
            sort=alt.EncodingSortField(
                field="mean_score_diff", op="max", order="descending"
            ),
        ),
        y=alt.Y("weighted_mean_score:Q"),
        text=alt.Text("mean_score_diff:Q", format=".3f"),
    )
)

# Add vertical line between 3rd and 4th blocks using pixel coordinates
vertical_line = (
    alt.Chart(pd.DataFrame({"x": [1]}))
    .mark_rule(color="gray", strokeDash=[5, 5], strokeWidth=2)
    .encode(
        x=alt.value(
            300
        ),  # Adjust this pixel value to position the line (chart width is 600)
        y=alt.value(0),
        y2=alt.value(400),
    )
)

# Combine the chart, text, and vertical line
final_chart = chart + text + vertical_line
final_chart

In [ ]:
# save HTML to upload dir
final_chart.save(OUT_PATH / "chart2.html")

### Recombine blocks with spatial

To show them on a map.

In [ ]:
# reset - combine top 3 and bottom 3
top_three = pd.DataFrame(block_means.head(6))
top_three["class"] = "top_three"
bottom_three = pd.DataFrame(block_means.tail(6))
bottom_three["class"] = "bottom_three"
block_means_short = pd.concat([top_three, bottom_three])

In [ ]:
# merge on geoid
block_means_short = block_means_short.merge(
    somerville_census_blocks[["GEOID20", "geometry"]],
    left_on="from_block_geoid",
    right_on="GEOID20",
    how="left",
)

# we only need one row for each block (eg, risk, not risk & length)
block_means_short[block_means_short["method"] == "risk"]

In [ ]:
block_means_short = gpd.GeoDataFrame(block_means_short)

In [ ]:
block_means_short.to_file(OUT_PATH / "block_means_short.gpkg", driver="GPKG")